# Baseline Text Classification Models

This notebook establishes classical machine learning baselines for primary ticket category classification.

The experiments compare word-level and character-level TF-IDF representations with Logistic Regression and Linear SVM.

Evaluation focuses on Macro-F1, weighted F1, accuracy, per-class performance, and confusion matrices.

The goal is to identify a strong classical baseline before evaluating more advanced multilingual transformer models.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

train_df = pd.read_csv(RAW_DATA_DIR / "train.csv")
validation_df = pd.read_csv(RAW_DATA_DIR / "validation.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (4000, 8)
Validation shape: (800, 8)


In [2]:
X_train = train_df["text"]
y_train = train_df["category"]

X_validation = validation_df["text"]
y_validation = validation_df["category"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))
print("Number of classes:", y_train.nunique())
print("Classes:", sorted(y_train.unique()))

Training samples: 4000
Validation samples: 800
Number of classes: 11
Classes: ['account_promo', 'app_technical', 'delivery_delay', 'food_quality', 'general_inquiry', 'lost_item', 'order_missing_wrong', 'payment_refund', 'ride_trip_issue', 'safety_conduct', 'spam_irrelevant']


In [3]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def evaluate_model(model, X, y):
    predictions = model.predict(X)

    accuracy = accuracy_score(y, predictions)
    macro_f1 = f1_score(y, predictions, average="macro")
    weighted_f1 = f1_score(y, predictions, average="weighted")

    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Macro-F1:    {macro_f1:.4f}")
    print(f"Weighted-F1: {weighted_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y, predictions))

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "predictions": predictions
    }

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

word_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

word_lr.fit(X_train, y_train)

word_lr_results = evaluate_model(
    word_lr,
    X_validation,
    y_validation
)

Accuracy:    0.4425
Macro-F1:    0.4300
Weighted-F1: 0.4154

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.58      0.26      0.36        72
      app_technical       0.77      0.16      0.26        64
     delivery_delay       0.38      0.40      0.39       120
       food_quality       0.62      0.14      0.23        56
    general_inquiry       0.87      0.33      0.47        40
          lost_item       0.71      0.12      0.21        40
order_missing_wrong       0.34      0.22      0.27        96
     payment_refund       0.33      0.87      0.48       144
    ride_trip_issue       0.62      0.64      0.63       112
     safety_conduct       0.90      0.28      0.43        32
    spam_irrelevant       1.00      1.00      1.00        24

           accuracy                           0.44       800
          macro avg       0.65      0.40      0.43       800
       weighted avg       0.54      0.44      0.42       800

In [5]:
from sklearn.svm import LinearSVC

# Use Linear SVM with the same word TF-IDF features for a fair comparison
word_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "classifier",
        LinearSVC(
            random_state=42
        )
    )
])

word_svm.fit(X_train, y_train)

word_svm_results = evaluate_model(
    word_svm,
    X_validation,
    y_validation
)

Accuracy:    0.5537
Macro-F1:    0.5906
Weighted-F1: 0.5467

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.74      0.56      0.63        72
      app_technical       0.79      0.36      0.49        64
     delivery_delay       0.51      0.42      0.46       120
       food_quality       0.41      0.21      0.28        56
    general_inquiry       0.60      0.65      0.63        40
          lost_item       0.84      0.78      0.81        40
order_missing_wrong       0.33      0.30      0.32        96
     payment_refund       0.43      0.83      0.57       144
    ride_trip_issue       0.73      0.62      0.67       112
     safety_conduct       0.80      0.62      0.70        32
    spam_irrelevant       0.89      1.00      0.94        24

           accuracy                           0.55       800
          macro avg       0.64      0.58      0.59       800
       weighted avg       0.59      0.55      0.55       800

In [6]:
# Use character n-grams to capture spelling, transliteration, and multilingual patterns
char_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            lowercase=True,
            ngram_range=(3, 5),
            min_df=3
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

char_lr.fit(X_train, y_train)

char_lr_results = evaluate_model(
    char_lr,
    X_validation,
    y_validation
)

Accuracy:    0.5100
Macro-F1:    0.4927
Weighted-F1: 0.4843

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.72      0.47      0.57        72
      app_technical       0.64      0.11      0.19        64
     delivery_delay       0.44      0.63      0.52       120
       food_quality       1.00      0.12      0.22        56
    general_inquiry       1.00      0.38      0.55        40
          lost_item       0.62      0.12      0.21        40
order_missing_wrong       0.48      0.34      0.40        96
     payment_refund       0.37      0.78      0.50       144
    ride_trip_issue       0.63      0.73      0.68       112
     safety_conduct       1.00      0.44      0.61        32
    spam_irrelevant       1.00      0.96      0.98        24

           accuracy                           0.51       800
          macro avg       0.72      0.46      0.49       800
       weighted avg       0.62      0.51      0.48       800

In [7]:
# Use Linear SVM with character n-grams for a stronger character baseline
char_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            lowercase=True,
            ngram_range=(3, 5),
            min_df=3
        )
    ),
    (
        "classifier",
        LinearSVC(
            random_state=42
        )
    )
])

char_svm.fit(X_train, y_train)

char_svm_results = evaluate_model(
    char_svm,
    X_validation,
    y_validation
)

Accuracy:    0.6200
Macro-F1:    0.6503
Weighted-F1: 0.6228

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.80      0.72      0.76        72
      app_technical       0.86      0.38      0.52        64
     delivery_delay       0.56      0.62      0.59       120
       food_quality       0.86      0.34      0.49        56
    general_inquiry       0.93      0.70      0.80        40
          lost_item       0.32      0.55      0.41        40
order_missing_wrong       0.43      0.40      0.41        96
     payment_refund       0.49      0.73      0.58       144
    ride_trip_issue       0.84      0.78      0.81       112
     safety_conduct       0.96      0.72      0.82        32
    spam_irrelevant       0.92      1.00      0.96        24

           accuracy                           0.62       800
          macro avg       0.73      0.63      0.65       800
       weighted avg       0.67      0.62      0.62       800

In [8]:
from sklearn.pipeline import FeatureUnion

# Combine word-level and character-level TF-IDF representations
word_char_features = FeatureUnion([
    (
        "word_tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "char_tfidf",
        TfidfVectorizer(
            analyzer="char",
            lowercase=True,
            ngram_range=(3, 5),
            min_df=3
        )
    )
])

word_char_svm = Pipeline([
    (
        "features",
        word_char_features
    ),
    (
        "classifier",
        LinearSVC(
            random_state=42
        )
    )
])

word_char_svm.fit(X_train, y_train)

word_char_results = evaluate_model(
    word_char_svm,
    X_validation,
    y_validation
)

Accuracy:    0.6088
Macro-F1:    0.6437
Weighted-F1: 0.6103

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.81      0.60      0.69        72
      app_technical       0.90      0.44      0.59        64
     delivery_delay       0.59      0.59      0.59       120
       food_quality       0.70      0.34      0.46        56
    general_inquiry       0.86      0.75      0.80        40
          lost_item       0.51      0.70      0.59        40
order_missing_wrong       0.31      0.31      0.31        96
     payment_refund       0.47      0.74      0.58       144
    ride_trip_issue       0.85      0.76      0.80       112
     safety_conduct       0.88      0.69      0.77        32
    spam_irrelevant       0.83      1.00      0.91        24

           accuracy                           0.61       800
          macro avg       0.70      0.63      0.64       800
       weighted avg       0.65      0.61      0.61       800

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Use text and language as model inputs
train_features = train_df[["text", "language"]]
validation_features = validation_df[["text", "language"]]

# Combine character TF-IDF text features with one-hot encoded language
char_language_features = ColumnTransformer([
    (
        "char_tfidf",
        TfidfVectorizer(
            analyzer="char",
            lowercase=True,
            ngram_range=(3, 5),
            min_df=3
        ),
        "text"
    ),
    (
        "language",
        OneHotEncoder(handle_unknown="ignore"),
        ["language"]
    )
])

# Train Linear SVM using text features and language metadata
char_language_svm = Pipeline([
    (
        "features",
        char_language_features
    ),
    (
        "classifier",
        LinearSVC(
            random_state=42
        )
    )
])

char_language_svm.fit(
    train_features,
    y_train
)

char_language_results = evaluate_model(
    char_language_svm,
    validation_features,
    y_validation
)

Accuracy:    0.6138
Macro-F1:    0.6444
Weighted-F1: 0.6178

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.80      0.72      0.76        72
      app_technical       0.90      0.41      0.56        64
     delivery_delay       0.55      0.61      0.58       120
       food_quality       0.87      0.36      0.51        56
    general_inquiry       0.93      0.65      0.76        40
          lost_item       0.33      0.55      0.41        40
order_missing_wrong       0.41      0.39      0.40        96
     payment_refund       0.48      0.69      0.57       144
    ride_trip_issue       0.84      0.78      0.81       112
     safety_conduct       0.92      0.75      0.83        32
    spam_irrelevant       0.83      1.00      0.91        24

           accuracy                           0.61       800
          macro avg       0.71      0.63      0.64       800
       weighted avg       0.67      0.61      0.62       800

In [11]:
# Use text and channel as model inputs
train_features = train_df[["text", "channel"]]
validation_features = validation_df[["text", "channel"]]

# Combine character TF-IDF text features with one-hot encoded channel
char_channel_features = ColumnTransformer([
    (
        "char_tfidf",
        TfidfVectorizer(
            analyzer="char",
            lowercase=True,
            ngram_range=(3, 5),
            min_df=3
        ),
        "text"
    ),
    (
        "channel",
        OneHotEncoder(handle_unknown="ignore"),
        ["channel"]
    )
])

# Train Linear SVM using text features and channel metadata
char_channel_svm = Pipeline([
    (
        "features",
        char_channel_features
    ),
    (
        "classifier",
        LinearSVC(
            random_state=42
        )
    )
])

char_channel_svm.fit(
    train_features,
    y_train
)

char_channel_results = evaluate_model(
    char_channel_svm,
    validation_features,
    y_validation
)

Accuracy:    0.6188
Macro-F1:    0.6529
Weighted-F1: 0.6230

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.80      0.71      0.75        72
      app_technical       0.88      0.36      0.51        64
     delivery_delay       0.56      0.62      0.59       120
       food_quality       0.88      0.38      0.53        56
    general_inquiry       0.94      0.72      0.82        40
          lost_item       0.32      0.55      0.40        40
order_missing_wrong       0.43      0.40      0.41        96
     payment_refund       0.48      0.72      0.58       144
    ride_trip_issue       0.85      0.78      0.81       112
     safety_conduct       0.92      0.72      0.81        32
    spam_irrelevant       0.96      1.00      0.98        24

           accuracy                           0.62       800
          macro avg       0.73      0.63      0.65       800
       weighted avg       0.67      0.62      0.62       800

In [12]:
# Combine subject and text into a single text input
train_features = train_df[["subject", "text", "channel"]].copy()
validation_features = validation_df[["subject", "text", "channel"]].copy()

train_features["subject"] = train_features["subject"].fillna("")
validation_features["subject"] = validation_features["subject"].fillna("")

train_features["combined_text"] = (
    train_features["subject"] + " " + train_features["text"]
)

validation_features["combined_text"] = (
    validation_features["subject"] + " " + validation_features["text"]
)

# Combine character TF-IDF with channel metadata
subject_text_channel_features = ColumnTransformer([
    (
        "char_tfidf",
        TfidfVectorizer(
            analyzer="char",
            lowercase=True,
            ngram_range=(3, 5),
            min_df=3
        ),
        "combined_text"
    ),
    (
        "channel",
        OneHotEncoder(handle_unknown="ignore"),
        ["channel"]
    )
])

# Train Linear SVM using subject, text, and channel
subject_text_channel_svm = Pipeline([
    (
        "features",
        subject_text_channel_features
    ),
    (
        "classifier",
        LinearSVC(
            random_state=42
        )
    )
])

subject_text_channel_svm.fit(
    train_features,
    y_train
)

subject_text_channel_results = evaluate_model(
    subject_text_channel_svm,
    validation_features,
    y_validation
)

Accuracy:    0.6238
Macro-F1:    0.6545
Weighted-F1: 0.6251

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.79      0.74      0.76        72
      app_technical       0.85      0.34      0.49        64
     delivery_delay       0.59      0.65      0.62       120
       food_quality       0.88      0.38      0.53        56
    general_inquiry       0.89      0.80      0.84        40
          lost_item       0.32      0.55      0.41        40
order_missing_wrong       0.40      0.38      0.39        96
     payment_refund       0.50      0.70      0.58       144
    ride_trip_issue       0.83      0.78      0.80       112
     safety_conduct       0.85      0.72      0.78        32
    spam_irrelevant       1.00      1.00      1.00        24

           accuracy                           0.62       800
          macro avg       0.72      0.64      0.65       800
       weighted avg       0.67      0.62      0.63       800

In [13]:
# Get B8 predictions and decision scores for error analysis
b8_predictions = subject_text_channel_svm.predict(validation_features)
b8_scores = subject_text_channel_svm.decision_function(validation_features)

print("Validation predictions:", len(b8_predictions))
print("Decision score shape:", b8_scores.shape)

Validation predictions: 800
Decision score shape: (800, 11)


In [14]:
# Create a table containing the actual and predicted categories
error_analysis_df = validation_features.copy()
error_analysis_df["actual_category"] = y_validation.values
error_analysis_df["predicted_category"] = b8_predictions

errors = error_analysis_df[
    error_analysis_df["actual_category"] != error_analysis_df["predicted_category"]
].copy()

print("Total validation samples:", len(error_analysis_df))
print("Misclassified samples:", len(errors))
print("Error rate:", len(errors) / len(error_analysis_df))

Total validation samples: 800
Misclassified samples: 301
Error rate: 0.37625


In [15]:
# Find the category pairs that are confused most often
confusion_pairs = (
    errors.groupby(
        ["actual_category", "predicted_category"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confusion_pairs.head(15)

,actual_category,predicted_category,count
34,order_missing_wrong,payment_refund,27
38,payment_refund,lost_item,25
15,food_quality,delivery_delay,19
13,delivery_delay,payment_refund,19
12,delivery_delay,order_missing_wrong,18
33,order_missing_wrong,lost_item,17
7,app_technical,order_missing_wrong,15
46,ride_trip_issue,payment_refund,14
2,account_promo,payment_refund,14
31,order_missing_wrong,delivery_delay,12


In [16]:
# Inspect representative examples from the most common confusion pairs
pairs_to_inspect = [
    ("order_missing_wrong", "payment_refund"),
    ("payment_refund", "lost_item"),
    ("food_quality", "delivery_delay")
]

for actual, predicted in pairs_to_inspect:
    print("=" * 80)
    print(f"ACTUAL: {actual}  |  PREDICTED: {predicted}")
    print("=" * 80)

    pair_errors = errors[
        (errors["actual_category"] == actual) &
        (errors["predicted_category"] == predicted)
    ]

    for _, row in pair_errors.head(5).iterrows():
        print(f"\nSubject: {row['subject']}")
        print(f"Channel: {row['channel']}")
        print(f"Text: {row['text']}")
        print("-" * 80)

ACTUAL: order_missing_wrong  |  PREDICTED: payment_refund

Subject: 
Channel: chat
Text: Here are the details of my request.
I have kept a copy of the confirmation.
The record I can see is SIMD-N5XCHYN.
I am using my usual device for this request.
Our delivery includes a main dish we never selected.
Can the support desk follow up?
--------------------------------------------------------------------------------

Subject: My service request
Channel: email
Text: Dear RideEat support,

pls advise on this situation.
The promised combo arrived without its accompanying beverage.
I do not remember the exact minute I opened the service, but I have the ref.
I have kept the confrmation rather than deleting it from the device.
I am giving as much background as I can remember so the record can be located.
I checked that this is the support address shown inside the app.
If you need another detail about the booking, pls let me know which one.
I am using my usual device for this request.
Can the suppo

In [17]:
# Inspect payment_refund tickets predicted as lost_item
pair_errors = errors[
    (errors["actual_category"] == "payment_refund") &
    (errors["predicted_category"] == "lost_item")
]

for _, row in pair_errors.head(10).iterrows():
    print("=" * 80)
    print(f"Subject: {row['subject']}")
    print(f"Channel: {row['channel']}")
    print(f"Text: {row['text']}")

Subject: විස්තර පරීක්ෂා කරන්න
Channel: email
Text: මට මේක බලලා දෙන්න පුළුවන්ද? එක පාරටම විස්තර කියන්න ටිකක් අමාරුයි. මම කලින් පණිවිඩ ටිකත් බලලා මේක ලියන්නේ. මම පණිවිඩය ලියන අතරේ වෙන වැඩකට යන්න වුණා. මොන විකාරයක්ද මේ!!! ඉතාම නරක සේවාවක්! රිසිට් එකේ ගානට වඩා වැඩිපුර අය කරලා. යොමු අංකය හොයන්න ටිකක් වෙලා ගියා. පිළිතුරක් දෙන්න.
Subject: 
Channel: chat
Text: කරුණාකර මේක බලන්න.
රිසිට් එකේ ගානට වඩා වැඩිපුර අය කරලා.
මගේ පවුලේ කෙනෙක් පණිවිඩය ලියන්න උදව් කළා.
ගෙවීම් සටහනේ රු. 1500 කියලා තියෙනවා.
තහවුරු කිරීමේ විස්තර මම තියාගෙන ඉන්නවා.
ඊළඟට මොකක්ද කරන්නේ?
Subject: 
Channel: chat
Text: meka therum ganna ba mama phone phone eken baluwe receipt eke ganata wada wadipura aran mama balagena inne 🤦
Subject: මේ ගැන බලන්න
Channel: email
Text: මගේ  ඉල්ලීම බලන්න
එක
පාරටම විස්තර කියන්න ටිකක් අමාරුයි
මගේ    ගෙදර  අයගෙන් ඇහුවට පස්සේ ආයෙත් විස්තර කියවලා බැලුවා    රිසිට් එකේ ගානට වඩා වැඩිපුර
අය කරලා
තහවුරු
කිරීමේ විස්තර මම තියාගෙන ඉන්නවා මම ආයෙත් විස්තර  පරීක්ෂා  කළා මේ  ගැන හොයලා  දැනුම් දෙන්න
Subject: 
Channel:

In [18]:
# Train the best feature setup with balanced class weights
balanced_svm = Pipeline([
    (
        "features",
        subject_text_channel_features
    ),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced",
            random_state=42
        )
    )
])

balanced_svm.fit(
    train_features,
    y_train
)

balanced_svm_results = evaluate_model(
    balanced_svm,
    validation_features,
    y_validation
)

Accuracy:    0.6312
Macro-F1:    0.6550
Weighted-F1: 0.6346

Classification Report:
                     precision    recall  f1-score   support

      account_promo       0.78      0.79      0.79        72
      app_technical       0.80      0.38      0.51        64
     delivery_delay       0.61      0.65      0.63       120
       food_quality       0.87      0.46      0.60        56
    general_inquiry       0.63      0.80      0.70        40
          lost_item       0.30      0.62      0.41        40
order_missing_wrong       0.45      0.39      0.42        96
     payment_refund       0.53      0.63      0.58       144
    ride_trip_issue       0.92      0.78      0.84       112
     safety_conduct       0.75      0.75      0.75        32
    spam_irrelevant       0.96      1.00      0.98        24

           accuracy                           0.63       800
          macro avg       0.69      0.66      0.66       800
       weighted avg       0.67      0.63      0.63       800

##  XLM-RoBERTa

This experiment fine-tunes a multilingual XLM-RoBERTa model for primary ticket category classification.

The model is evaluated using the same metrics as the classical baselines, with Macro-F1 as the primary metric.

In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Define a stable label mapping for the 11 primary categories
labels = sorted(y_train.unique())

label2id = {
    label: idx
    for idx, label in enumerate(labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print("Number of classes:", len(labels))
print("Label mapping:")
for idx, label in id2label.items():
    print(idx, "->", label)

Number of classes: 11
Label mapping:
0 -> account_promo
1 -> app_technical
2 -> delivery_delay
3 -> food_quality
4 -> general_inquiry
5 -> lost_item
6 -> order_missing_wrong
7 -> payment_refund
8 -> ride_trip_issue
9 -> safety_conduct
10 -> spam_irrelevant


In [20]:
# Load the multilingual tokenizer used by XLM-RoBERTa
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizer: xlm-roberta-base
Vocabulary size: 250002


In [21]:
sample_texts = train_df["text"].iloc[:3].tolist()

for i, text in enumerate(sample_texts, start=1):
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=256
    )

    print(f"Sample {i}")
    print("Original:", text[:150])
    print("Token count:", len(tokens["input_ids"]))
    print("Tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"])[:20])
    print()

Sample 1
Original: I selected the service earlier today. I need help with the following situation. The fare was supposed to be LKR 3050, but a different sum was collecte
Token count: 46
Tokens: ['<s>', '▁I', '▁selected', '▁the', '▁service', '▁earlier', '▁today', '.', '▁I', '▁need', '▁help', '▁with', '▁the', '▁following', '▁situation', '.', '▁The', '▁fare', '▁was', '▁supposed']

Sample 2
Original: The screen shows the reference SIMQ-V9LU2KD
Can you check the details below
The packet arrived but half the items on the receipt are absent
Can you he
Token count: 43
Tokens: ['<s>', '▁The', '▁screen', '▁shows', '▁the', '▁reference', '▁SIM', 'Q', '-', 'V', '9', 'LU', '2', 'K', 'D', '▁Can', '▁you', '▁check', '▁the', '▁details']

Sample 3
Original: uh hello agent: rideeat  call karata  thanks mokakda
prashne? customer:  me gana kiyannai  liwwe. details hoyala

liyanna tikak wela giya. agent: thaw
Token count: 123
Tokens: ['<s>', '▁uh', '▁hell', 'o', '▁agent', ':', '▁ride', 'e', 'at', '▁call', '▁

In [22]:
token_lengths = train_df["text"].apply(
    lambda text: len(
        tokenizer(
            text,
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )
)

print(token_lengths.describe(percentiles=[0.90, 0.95, 0.99]))

count    4000.000000
mean       79.106500
std        33.604196
min        15.000000
50%        71.000000
90%       127.000000
95%       139.000000
99%       169.010000
max       223.000000
Name: text, dtype: float64


In [24]:
from datasets import Dataset

# Keep only text and primary category
train_model_df = train_df[["text", "category"]].copy()
validation_model_df = validation_df[["text", "category"]].copy()

# Convert category labels to numeric IDs
train_model_df["label"] = train_model_df["category"].map(label2id)
validation_model_df["label"] = validation_model_df["category"].map(label2id)

# Convert pandas DataFrames to Hugging Face datasets
train_dataset = Dataset.from_pandas(
    train_model_df[["text", "label"]],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_model_df[["text", "label"]],
    preserve_index=False
)

print(train_dataset)
print(validation_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 4000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 800
})


In [25]:
MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_dataset.map(
    tokenize_batch,
    batched=True
)

validation_tokenized = validation_dataset.map(
    tokenize_batch,
    batched=True
)

print(train_tokenized)
print(validation_tokenized)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 4000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 800
})


In [26]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)

print("Model:", MODEL_NAME)
print("Number of labels:", model.config.num_labels)
print("Model type:", model.config.model_type)

C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: xlm-roberta-base
Number of labels: 11
Model type: xlm-roberta


In [27]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory: 6.0 GB


In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/xlmr_primary",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    seed=42
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start

In [29]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    weighted_f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer created successfully.")

Trainer created successfully.


In [33]:
import torch

model.to("cuda")
model.train()

batch = train_tokenized.select(range(4))

batch = {
    key: torch.tensor(batch[key]).to("cuda")
    for key in ["input_ids", "attention_mask"]
}

batch["labels"] = torch.tensor(
    train_tokenized.select(range(4))["label"]
).to("cuda")

outputs = model(**batch)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

Loss: 2.4406967163085938
Logits shape: torch.Size([4, 11])


In [34]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,3.955058,1.196453,0.596250,0.465637,0.552812
2,1.141038,0.874793,0.767500,0.705354,0.753403
3,0.441360,0.766744,0.796250,0.788575,0.797893


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
# Import evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

import numpy as np

# Generate predictions on the validation dataset
predictions_output = trainer.predict(validation_tokenized)

# Convert model logits into predicted class indices
predictions = np.argmax(
    predictions_output.predictions,
    axis=-1
)

# Get the true class labels
true_labels = predictions_output.label_ids

# Calculate overall evaluation metrics
print(
    "Accuracy:",
    accuracy_score(true_labels, predictions)
)

print(
    "Macro-F1:",
    f1_score(
        true_labels,
        predictions,
        average="macro"
    )
)

print(
    "Weighted-F1:",
    f1_score(
        true_labels,
        predictions,
        average="weighted"
    )
)

# Display performance for each primary category
print("\nClassification Report:")

print(
    classification_report(
        true_labels,
        predictions,
        target_names=labels,
        digits=4
    )
)

Accuracy: 0.79625
Macro-F1: 0.7885747870653058
Weighted-F1: 0.7978926474156074

Classification Report:
                     precision    recall  f1-score   support

      account_promo     0.9200    0.9583    0.9388        72
      app_technical     0.8095    0.7969    0.8031        64
     delivery_delay     0.7794    0.8833    0.8281       120
       food_quality     0.8039    0.7321    0.7664        56
    general_inquiry     0.6792    0.9000    0.7742        40
          lost_item     1.0000    0.7750    0.8732        40
order_missing_wrong     0.5503    0.8542    0.6694        96
     payment_refund     0.9677    0.6250    0.7595       144
    ride_trip_issue     0.8839    0.8839    0.8839       112
     safety_conduct     0.9444    0.5312    0.6800        32
    spam_irrelevant     0.7895    0.6250    0.6977        24

           accuracy                         0.7963       800
          macro avg     0.8298    0.7786    0.7886       800
       weighted avg     0.8302    0.7963 

In [36]:
# Calculate the confusion matrix
cm = confusion_matrix(true_labels, predictions)

# Create a readable DataFrame with class names
cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

# Display the confusion matrix
cm_df

,account_promo,app_technical,delivery_delay,food_quality,general_inquiry,lost_item,order_missing_wrong,payment_refund,ride_trip_issue,safety_conduct,spam_irrelevant
account_promo,69,2,1,0,0,0,0,0,0,0,0
app_technical,5,51,7,0,0,0,0,1,0,0,0
delivery_delay,0,0,106,0,1,0,11,2,0,0,0
food_quality,0,0,0,41,0,0,15,0,0,0,0
general_inquiry,0,0,4,0,36,0,0,0,0,0,0
lost_item,0,8,0,0,0,31,0,0,1,0,0
order_missing_wrong,0,0,4,3,3,0,82,0,0,0,4
payment_refund,0,0,5,6,2,0,41,90,0,0,0
ride_trip_issue,1,2,7,0,2,0,0,0,99,1,0
safety_conduct,0,0,2,1,0,0,0,0,12,17,0


In [37]:
# Create a DataFrame containing the validation texts and predictions
error_df = validation_df.copy()

error_df["predicted_category"] = [
    id2label[pred]
    for pred in predictions
]

# Find the largest confusion pair
payment_order_errors = error_df[
    (error_df["category"] == "payment_refund") &
    (error_df["predicted_category"] == "order_missing_wrong")
]

# Display the examples
print("Number of errors:", len(payment_order_errors))

payment_order_errors[
    ["text", "language", "channel", "category", "predicted_category"]
]

Number of errors: 41


,text,language,channel,category,predicted_category
0,මට මේක බලලා දෙන්න පුළුවන්ද? එක පාරටම විස්තර කි...,si,email,payment_refund,order_missing_wrong
7,um um rideeat call karata thanks mokakda prash...,singlish,call_transcript,payment_refund,order_missing_wrong
9,මට මේක බලලා දෙන්න පුළුවන්ද? එක පාරටම විස්තර කි...,si,email,payment_refund,order_missing_wrong
65,පොඩ්ඩක් උදව් කරනවද?\nරිසිට් එකේ ගානට වඩා වැඩිප...,mixed,chat,payment_refund,order_missing_wrong
83,agent: RideEat ku call pannathukku thanks enna...,tanglish,call_transcript,payment_refund,order_missing_wrong
97,agent: RideEat call karata thanks mokakda pras...,singlish,call_transcript,payment_refund,order_missing_wrong
102,poddk help karanawada?\n\nokkom detals ek para...,singlish,email,payment_refund,order_missing_wrong
108,meka therum ganna ba mama phone phone eken bal...,singlish,chat,payment_refund,order_missing_wrong
137,um um \nrideeat call karata thanks mokakda ...,singlish,call_transcript,payment_refund,order_missing_wrong
140,help ekak\nona receipt eke ganata wada wad...,singlish,chat,payment_refund,order_missing_wrong


In [38]:
import re

# Normalize text to reveal near-duplicate templates
def normalize_for_template(text):
    text = str(text).lower()

    # Remove reference IDs and numbers
    text = re.sub(r"\b[a-z0-9]+-[a-z0-9-]+\b", " ", text)
    text = re.sub(r"\d+", " ", text)

    # Remove speaker labels
    text = re.sub(r"\b(agent|customer)\s*:", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Create normalized text for the validation set
error_df["normalized_text"] = error_df["text"].apply(
    normalize_for_template
)

# Count repeated normalized templates
template_counts = (
    error_df["normalized_text"]
    .value_counts()
)

# Show the most repeated templates
template_counts.head(20)

normalized_text
dear rideeat support, i am contacting you for help today. i have kept a copy of the confirmation. there were several messages on the screen and i read through them before contacting support. i am giving as much background as i can remember so the record can be located. i am trying to keep the description clear although i may have left out some background. if you need another detail about the booking, please let me know which one. the promised combo arrived without its accompanying beverage. please advise on the next step. regards, a customer                                                                                                                                                                                                                      1
ஒரு உதவி வேணும். நினைவில் உள்ளபடி முதலில் தொலைபேசியில் உறுதிப்படுத்தலைப் பார்த்தேன். விவரங்களைத் தேடி எழுதச் சிறிது நேரம் எடுத்தது. எல்லா விவரங்களையும் ஒரே தடவையில் சொல்லச் சிரமமாக உள்ளது. ரசீதில் உள்ள தொகையை விட அதிகமாக 

In [39]:
# Get normalized templates from the payment -> order errors
payment_order_templates = (
    payment_order_errors["text"]
    .apply(normalize_for_template)
    .value_counts()
)

print("Unique normalized templates:",
      len(payment_order_templates))

print("\nMost repeated templates:")
print(payment_order_templates.head(20))

Unique normalized templates: 41

Most repeated templates:
text
මට මේක බලලා දෙන්න පුළුවන්ද? එක පාරටම විස්තර කියන්න ටිකක් අමාරුයි. මම කලින් පණිවිඩ ටිකත් බලලා මේක ලියන්නේ. මම පණිවිඩය ලියන අතරේ වෙන වැඩකට යන්න වුණා. මොන විකාරයක්ද මේ!!! ඉතාම නරක සේවාවක්! රිසිට් එකේ ගානට වඩා වැඩිපුර අය කරලා. යොමු අංකය හොයන්න ටිකක් වෙලා ගියා. පිළිතුරක් දෙන්න.                                                                                                                                                                                                                                                                    1
um um rideeat call karata thanks mokakda prashne poddak help karanawada detals hoyala [inaudible] liyanna tikak wela giya thawa detals kiyanna receipt eke ganata wada wadipura wadipura aran screeen eke screenshot ekak thiyagaththa reference eka hoyanna tikak wela giya me gana hoyala kiyanna                                                                                                               

In [40]:
from sklearn.metrics import f1_score

# Add predictions to the validation data
language_eval_df = validation_df.copy()

language_eval_df["predicted_category"] = [
    id2label[pred]
    for pred in predictions
]

# Calculate Macro-F1 separately for each language
language_results = []

for language in sorted(language_eval_df["language"].unique()):
    language_data = language_eval_df[
        language_eval_df["language"] == language
    ]

    macro_f1 = f1_score(
        language_data["category"],
        language_data["predicted_category"],
        average="macro"
    )

    accuracy = accuracy_score(
        language_data["category"],
        language_data["predicted_category"]
    )

    language_results.append({
        "language": language,
        "samples": len(language_data),
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

language_results_df = pd.DataFrame(language_results)

language_results_df

,language,samples,accuracy,macro_f1
0,en,280,0.839286,0.821544
1,mixed,20,0.750000,0.659740
2,si,160,0.725000,0.668456
3,singlish,200,0.735000,0.735359
4,ta,120,0.908333,0.805025
5,tanglish,20,0.750000,0.828283


In [41]:
#Calculate class weights
from sklearn.utils.class_weight import compute_class_weight

# Calculate balanced weights for each primary category
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(labels)),
    y=train_model_df["label"]
)

# Convert weights to a PyTorch tensor
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
)

# Display the weight assigned to each class
for idx, label in enumerate(labels):
    print(f"{label}: {class_weights[idx]:.3f}")

account_promo: 1.010
app_technical: 1.136
delivery_delay: 0.606
food_quality: 1.299
general_inquiry: 1.818
lost_item: 1.818
order_missing_wrong: 0.758
payment_refund: 0.505
ride_trip_issue: 0.649
safety_conduct: 2.273
spam_irrelevant: 3.030


In [58]:
import gc
import torch

del weighted_trainer
del weighted_model

gc.collect()
torch.cuda.empty_cache()

print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Allocated: 7.20 GB
Reserved: 8.15 GB


In [59]:
weighted_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)

weighted_model.to("cuda")

Loading weights:   0%|          | 0/197 [00:01<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=11, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): L

In [60]:
weighted_training_args = TrainingArguments(
    output_dir="../models/xlmr_primary_weighted",

    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=True,
    gradient_checkpointing=True,

    optim="adafactor",

    logging_strategy="steps",
    logging_steps=50,

    report_to="none",
    seed=42
)

In [61]:
weighted_trainer = WeightedTrainer(
    model=weighted_model,
    args=weighted_training_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [63]:
#weighted_train_result = weighted_trainer.train()